# Параллельные вычисления

Материалы:
* Макрушин С.В. Лекция 10: Параллельные вычисления
* https://docs.python.org/3/library/multiprocessing.html

## Задачи для совместного разбора

1. Посчитайте, сколько раз встречается каждый из символов (заглавные и строчные символы не различаются) в файле `Dostoevskiy Fedor. Prestuplenie i nakazanie - BooksCafe.Net.txt` и в файле `Dostoevskiy Fedor. Igrok - BooksCafe.Net.txt`.

In [6]:
from collections import Counter

In [12]:
with open('Dostoevskiy Fedor. Prestuplenie i nakazanie - BooksCafe.Net.txt', encoding='cp1251') as f:
  lines = f.readlines()

  for line in zip(lines, range(5)):
    print(line)

('Спасибо, что скачали книгу в бесплатной электронной библиотеке BooksCafe.Net: http://bookscafe.net\n', 0)
('\n', 1)
('Все книги автора: http://bookscafe.net/author/dostoevskiy_fedor-1096.html\n', 2)
('\n', 3)
('Эта же книга в других форматах: http://bookscafe.net/book/dostoevskiy_fedor-prestuplenie_i_nakazanie-219855.html\n', 4)


2. Решить задачу 1, распараллелив вычисления с помощью модуля `multiprocessing`. Для обработки каждого файла создать свой собственный процесс.

## Лабораторная работа 10

In [1]:
import csv
import multiprocessing
import pandas as pd

import ast

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
%cd /content/drive/MyDrive/ФУППРФ/5 семестр/Технологии обработки больших данных/data

/content/drive/MyDrive/ФУППРФ/5 семестр/Технологии обработки больших данных/data


1. Разбейте файл `recipes_full.csv` на несколько (например, 8) примерно одинаковых по объему файлов c названиями `id_tag_nsteps_*.csv`. Каждый файл содержит 3 столбца: `id`, `tag` и `n_steps`, разделенных символом `;`. Для разбора строк используйте `csv.reader`.

__Важно__: вы не можете загружать в память весь файл сразу. Посмотреть на первые несколько строк файла вы можете, написав код, который считывает эти строки.

Подсказка: примерное кол-во строк в файле - 2.3 млн.

```
id;tag;n_steps
137739;60-minutes-or-less;11
137739;time-to-make;11
137739;course;11
```


In [15]:
n_rows = 2300000
n_files = 30
rows_per_file = n_rows // (n_files - 1)

In [ ]:
with open('recipes_full.csv', newline='') as f:
  reader = csv.reader(f, delimiter=',')
  for line, _ in zip(reader, range(10)):
    print(line[5])

['1279780;tomatoes;2']
['1279780;turkish;2']
['1279780;vegetarian;2']
['1279780;pork-loin;2']
['1279780;granola-and-porridge;2']
['1279780;short-grain-rice;2']
['1279780;less_thansql:name_topics_of_recipegreater_than;2']
['1279780;grilling;2']
['667373;high-protein;1']
['667373;crawfish;1']


In [ ]:
with open('recipes_full.csv', newline='') as f:
  reader = csv.reader(f, delimiter=',')

  for file_ind in range(n_files):
    with open(f'id_tag_nsteps_{file_ind}.csv', 'w', newline='') as csvfile:
      writer = csv.writer(csvfile, delimiter=';')
      writer.writerow(["id", "tag", "n_steps"])
      for row, _ in zip(reader, range(rows_per_file)):
        id_ = row[1]
        tags = row[5]
        n_steps = row[6]
        if tags == 'tags':
          continue
        for tag in list(ast.literal_eval(tags)):
          writer.writerow([id_, tag, n_steps])


2. Напишите функцию, которая принимает на вход название файла, созданного в результате решения задачи 1, считает среднее значение количества шагов для каждого тэга и возвращает результат в виде словаря.

In [4]:
def n_steps_per_tag(file_name):
  df = pd.read_csv(file_name, delimiter=';')
  return df[['tag', 'n_steps']].groupby('tag').mean().to_dict()

In [5]:
pd.read_csv('id_tag_nsteps_0.csv', delimiter=';').head()

,id,tag,n_steps
0,683970,mexican,4
1,683970,healthy-2,4
2,683970,orange-roughy,4
3,683970,chicken-thighs-legs,4
4,683970,freezer,4


3. Напишите функцию, которая считает среднее значение количества шагов для каждого тэга по всем файлам, полученным в задаче 1, и возвращает результат в виде словаря. Не используйте параллельных вычислений. При реализации выделите функцию, которая объединяет результаты обработки отдельных файлов. Модифицируйте код из задачи 2 таким образом, чтобы иметь возможность получить результат, имея результаты обработки отдельных файлов. Определите, за какое время задача решается для всех файлов.


In [16]:
from functools import reduce
def n_steps_per_tag_all():
  return reduce(combine_results, [f'id_tag_nsteps_{file_ind}.csv' for file_ind in range(1, n_files)], pd.DataFrame(n_steps_per_tag('id_tag_nsteps_0.csv'))).T.mean()

def combine_results(res, file):
  df = pd.DataFrame(n_steps_per_tag(file))
  return pd.concat([res, df], axis=1, ignore_index=True)

In [17]:
n_steps_per_tag_all().to_dict()

{'1-day-or-more': 4.499501547210346,
 '15-minutes-or-less': 4.980776949417512,
 '3-steps-or-less': 4.732122466300192,
 '30-minutes-or-less': 7.609138968909821,
 '4-hours-or-less': 10.06130321716521,
 '5-ingredients-or-less': 5.360052214430903,
 '60-minutes-or-less': 9.413565130281125,
 'Throw the ultimate fiesta with this sopaipillas recipe from Food.com.': 3.5189359733158945,
 'a1-sauce': 3.526801566285487,
 'african': 4.361174703779738,
 'american': 7.587817731869675,
 'amish-mennonite': 3.581011952728535,
 'angolan': 3.487828519767331,
 'appetizers': 6.248757341684779,
 'apples': 4.854144295206155,
 'april-fools-day': 3.5158204572390015,
 'argentine': 3.559288453283473,
 'artichoke': 3.4977734563076646,
 'asian': 6.446989670584447,
 'asparagus': 4.054111398021619,
 'australian': 4.210483389850024,
 'austrian': 3.5714112786152943,
 'avocado': 3.531117783003695,
 'bacon': 4.101907878932196,
 'baja': 3.547470627043662,
 'baked-beans': 3.4811633559192687,
 'baking': 3.6307083667126494,


4. Решите задачу 3, распараллелив вычисления с помощью модуля `multiprocessing`. Для обработки каждого файла создайте свой собственный процесс. Определите, за какое время задача решается для всех файлов.

In [18]:
import time
from functools import wraps
def timeit(func):
    @wraps(func)
    def timeit_wrapper(*args, **kwargs):
        start_time = time.perf_counter()
        result = func(*args, **kwargs)
        end_time = time.perf_counter()
        total_time = end_time - start_time
        print(f'For file {args[0]} it took {total_time:.4f} seconds\n')
        return result
    return timeit_wrapper

In [19]:
import multiprocessing as mp

In [20]:
@timeit
def n_steps_per_tag_mp(file_name, return_dict):
  df = pd.read_csv(file_name, delimiter=';')
  return_dict[file_name] = df[['tag', 'n_steps']].groupby('tag').mean().to_dict()

def combine_results_mp(res1, res2):
  return pd.concat([pd.DataFrame(res1), pd.DataFrame(res2)], axis=1, ignore_index=True)

In [21]:
files = [f'id_tag_nsteps_{file_ind}.csv' for file_ind in range(n_files)]

manager = mp.Manager()
return_dict = manager.dict()

jobs = []
for file_ in files:
  p = mp.Process(target=n_steps_per_tag_mp, args=[file_, return_dict])
  jobs.append(p)
  p.start()

for job in jobs:
  job.join()

For file id_tag_nsteps_29.csv it took 0.4057 seconds

For file id_tag_nsteps_28.csv it took 1.4258 seconds

For file id_tag_nsteps_2.csv it took 6.0787 seconds

For file id_tag_nsteps_4.csv it took 6.3618 seconds

For file id_tag_nsteps_1.csv it took 6.8138 seconds

For file id_tag_nsteps_3.csv it took 6.8721 seconds

For file id_tag_nsteps_0.csv it took 7.1490 seconds

For file id_tag_nsteps_5.csv it took 7.3158 seconds

For file id_tag_nsteps_7.csv it took 7.2962 seconds

For file id_tag_nsteps_6.csv it took 7.4021 seconds

For file id_tag_nsteps_8.csv it took 7.4526 seconds

For file id_tag_nsteps_12.csv it took 7.3055 seconds

For file id_tag_nsteps_11.csv it took 7.4225 seconds

For file id_tag_nsteps_9.csv it took 7.6926 seconds

For file id_tag_nsteps_10.csv it took 7.6808 seconds

For file id_tag_nsteps_13.csv it took 7.6597 seconds

For file id_tag_nsteps_16.csv it took 7.3966 seconds
For file id_tag_nsteps_14.csv it took 7.6805 seconds
For file id_tag_nsteps_17.csv it took 7.

In [22]:
reduce(combine_results_mp, return_dict.values()).T.mean().to_dict()

{'1-day-or-more': 4.499501547210346,
 '15-minutes-or-less': 4.980776949417511,
 '3-steps-or-less': 4.732122466300192,
 '30-minutes-or-less': 7.609138968909821,
 '4-hours-or-less': 10.061303217165209,
 '5-ingredients-or-less': 5.360052214430902,
 '60-minutes-or-less': 9.413565130281125,
 'Throw the ultimate fiesta with this sopaipillas recipe from Food.com.': 3.5189359733158945,
 'a1-sauce': 3.526801566285487,
 'african': 4.361174703779738,
 'american': 7.587817731869675,
 'amish-mennonite': 3.581011952728536,
 'angolan': 3.487828519767331,
 'appetizers': 6.2487573416847795,
 'apples': 4.8541442952061535,
 'april-fools-day': 3.515820457239002,
 'argentine': 3.559288453283473,
 'artichoke': 3.497773456307664,
 'asian': 6.446989670584447,
 'asparagus': 4.054111398021617,
 'australian': 4.210483389850024,
 'austrian': 3.5714112786152943,
 'avocado': 3.5311177830036953,
 'bacon': 4.101907878932196,
 'baja': 3.5474706270436616,
 'baked-beans': 3.4811633559192683,
 'baking': 3.630708366712648

5. (*) Решите задачу 3, распараллелив вычисления с помощью модуля `multiprocessing`. Создайте фиксированное количество процессов (равное половине количества ядер на компьютере). При помощи очереди передайте названия файлов для обработки процессам и при помощи другой очереди заберите от них ответы.

In [23]:
mp.cpu_count()

2

In [48]:
NUM_PROCESSES = 2

files_q = multiprocessing.JoinableQueue()
res_q = multiprocessing.JoinableQueue()



def f(queue, res_q):
  while True:
    file_name = queue.get(block=True)

    if file_name is None:
      break
    n_steps_per_tag_mpq(file_name, res_q)
    queue.task_done()
  queue.task_done()

def n_steps_per_tag_mpq(file_name, res_q):
  df = pd.read_csv(file_name, delimiter=';')
  res_q.put(df[['tag', 'n_steps']].groupby('tag').mean().to_dict())
  print(f'got res of handling {file_name}')


pool = mp.Pool(NUM_PROCESSES, f, (files_q, res_q))

for file_name in [f'id_tag_nsteps_{file_ind}.csv' for file_ind in range(n_files)]:
  files_q.put(file_name)
  print(f'put {file_name}')


for i in range(NUM_PROCESSES):
    files_q.put(None)

files_q.close()
files_q.join()



pool.close()
# pool.join()

results = []
while not res_q.empty():
  results.append(res_q.get(True))

put id_tag_nsteps_0.csv
put id_tag_nsteps_1.csv
put id_tag_nsteps_2.csv
put id_tag_nsteps_3.csv
put id_tag_nsteps_4.csv
put id_tag_nsteps_5.csv
put id_tag_nsteps_6.csv
put id_tag_nsteps_7.csv
put id_tag_nsteps_8.csv
put id_tag_nsteps_9.csv
put id_tag_nsteps_10.csv
put id_tag_nsteps_11.csv
put id_tag_nsteps_12.csv
put id_tag_nsteps_13.csv
put id_tag_nsteps_14.csv
put id_tag_nsteps_15.csv
put id_tag_nsteps_16.csv
put id_tag_nsteps_17.csv
put id_tag_nsteps_18.csv
put id_tag_nsteps_19.csv
put id_tag_nsteps_20.csv
put id_tag_nsteps_21.csv
put id_tag_nsteps_22.csv
put id_tag_nsteps_23.csv
put id_tag_nsteps_24.csv
put id_tag_nsteps_25.csv
put id_tag_nsteps_26.csv
put id_tag_nsteps_27.csv
put id_tag_nsteps_28.csv
put id_tag_nsteps_29.csv
got res of handling id_tag_nsteps_1.csv
got res of handling id_tag_nsteps_0.csv
got res of handling id_tag_nsteps_3.csv
got res of handling id_tag_nsteps_2.csv
got res of handling id_tag_nsteps_4.csvgot res of handling id_tag_nsteps_5.csv

got res of handling 

In [49]:
reduce(combine_results_mp, results).T.mean().to_dict()

{'1-day-or-more': 4.499501547210346,
 '15-minutes-or-less': 4.980776949417512,
 '3-steps-or-less': 4.732122466300193,
 '30-minutes-or-less': 7.609138968909821,
 '4-hours-or-less': 10.061303217165209,
 '5-ingredients-or-less': 5.360052214430903,
 '60-minutes-or-less': 9.413565130281125,
 'Throw the ultimate fiesta with this sopaipillas recipe from Food.com.': 3.5189359733158945,
 'a1-sauce': 3.5268015662854877,
 'african': 4.361174703779739,
 'american': 7.587817731869675,
 'amish-mennonite': 3.581011952728536,
 'angolan': 3.487828519767331,
 'appetizers': 6.2487573416847795,
 'apples': 4.854144295206155,
 'april-fools-day': 3.5158204572390015,
 'argentine': 3.559288453283473,
 'artichoke': 3.4977734563076646,
 'asian': 6.446989670584449,
 'asparagus': 4.054111398021618,
 'australian': 4.210483389850024,
 'austrian': 3.571411278615295,
 'avocado': 3.5311177830036953,
 'bacon': 4.101907878932196,
 'baja': 3.5474706270436616,
 'baked-beans': 3.4811633559192683,
 'baking': 3.63070836671264